# Suite TXN — Transactions: mutation expressions over frames, and what they are built from

A transaction is a mutation expression over a stream of frames (docs/STREAMING_QUERIES.md; docs/OBJECT_IDENTITY.md,
Decided, 85 to 97): its terms are reference objects of the `Mutations` schemas, linked by relations, each term's
pattern a `Schemas.OfPattern` built through its link, and a term chosen through a kind union, `Mutations.OfTerm`. A
pattern's `spec` says what must hold of a frame, written in the builders' notation over the frame's symbols and recorded
as mutations. A keyed adjacency takes a mapping in place of the peer shorthand (FRAMEWORK.md, Resolved).

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import { Builders as B, Paths, Plain, Schemas as S } from "@mbse/schemas/Framework";
import { AttributeError, LookupError, ReadOnlyError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, equal, map, raises, same } from "./support.js";

/** crm.Person, who knows persons since a year; crm.Book, a singleton owning its contacts by name; crm.Note, counted. */
function crm(store: any): any[] {
  const [Int, Str] = [store.Schemas.OfInt, store.Schemas.OfStr];
  const Knows = store.Schemas.OfRelation((r: any) => r.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")])
    .properties({ since: Int }));
  const Person = store.Schemas.OfObject((o: any) => o.name("crm.Person").ref(true).properties({ name: Str, age: Int }));
  const Holds = store.Schemas.OfRelation((r: any) => (
    r.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")]).properties({ key: Str })
      .uniques([["book", "key"], ["contact"]])));
  store.Schemas.OfObject(Person, (o: any) => o.adjacencies([
    (t: any) => t.name("knows").relation([Knows]).me("knower"),
    (t: any) => t.relation([Knows]).me("known"),
    (t: any) => t.name("book").relation([Holds]).me("contact").owner(true),
  ]));
  const Book = store.Schemas.OfObject((o: any) => o.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
    contacts: (t: any) => t.relation([Holds]).me("book") }));
  const Note = store.Schemas.OfObject((o: any) => o.name("crm.Note").ref(true).properties({ text: Str }));
  const Meets = store.Schemas.OfRelation((r: any) => r.name("crm.Meets").links(["a", "b", "c"].map((n) => (t: any) => t.name(n))));
  const Kind = store.Schemas.OfUnion((u: any) => u.name("crm.Kind").branches([(t: any) => t.name("person").schema([Person]),
    (t: any) => t.name("note").schema([Note])]));
  const Picks = store.Schemas.OfRelation((r: any) => r.name("crm.Picks").links([(t: any) => t.name("owner"),
    (t: any) => t.name("pick").schema([Kind])]).properties({ rank: Int }));
  const Pins = store.Schemas.OfRelation((r: any) => r.name("crm.Pins").links([(t: any) => t.name("note"),
    (t: any) => t.name("book").schema([Book])]).properties({ at: Int }));
  const Trio = store.Schemas.OfRelation((r: any) => r.links(["a", "b", "c"].map((n) => (t: any) => t.name(n)))); // unnamed
  const Odd = store.Schemas.OfObject((o: any) => o.name("crm.Odd").ref(true).adjacencies([(t: any) => t.name("x").me("a")])); // no relation yet
  const Odds = store.Schemas.OfRelation((r: any) => r.name("crm.Odds").links([(t: any) => t.name("note"), (t: any) => t.name("odd").schema([Odd])])
    .properties({ k: Int }));
  store.Schemas.OfObject(Person, (o: any) => o.adjacencies({ meets: (t: any) => t.relation([Meets]).me("a"),
    trio: (t: any) => t.relation([Trio]).me("a") }));
  store.Schemas.OfObject(Note, (o: any) => o.adjacencies({ picks: (t: any) => t.relation([Picks]).me("owner"),
    pins: (t: any) => t.relation([Pins]).me("note"), odds: (t: any) => t.relation([Odds]).me("note") }));
  return [Person, Knows, Book, Note];
}

const store: any = new B.Store();
const [Int, Str] = [store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()];
const [Person, Knows, Book, Note] = crm(store);

## TXN-01 · A kind union: a link's schema a union of reference object schemas, each branch a kind built through it

In [ ]:
{
  const Mutations = store.Mutations;
  const OfTerm = Mutations.OfTerm.as_schema();
  const kinds = S._kinds(OfTerm) as Map<string, unknown>;
  assert(same([...kinds.keys()], ["map", "reduce", "partition", "update", "upsert"])
    && [...kinds.values()].every((kind, i) => kind === [Mutations.OfMap, Mutations.OfReduce, Mutations.OfPartition,
      Mutations.OfUpdate, Mutations.OfUpsert][i].as_schema()));
  assert(S._kinds(Person) === null && S._kinds(store.Schemas.OfStr.as_schema()) === null);
  assert(store.names().filter((name: string) => name.startsWith("Mutations.")).every((name: string) => store.registered(name).validate().length === 0));
  const mixed = store.Schemas.OfUnion((u: any) => u.branches([(t: any) => t.name("p").schema([Person]),
    (t: any) => t.name("v").schema([store.Schemas.OfObject((o: any) => o)])]));
  assert(same(store.Schemas.OfRelation((r: any) => r.links([(t: any) => t.name("a"), (t: any) => t.name("b").schema([mixed])])).validate(),
    ["link 'b': a kind union's branches are reference object schemas"]));
  // Building through the link: a branch, called, builds an object of its kind; the peer shorthand's visitor and an entry
  // builder each have the branches as members.
  const tx = store.Mutations.OfTransaction((t: any) => t.uid("t-1").mutations([(mu: any) => mu.upsert((u: any) => (
    u.key("k").then([(w: any) => w.update((d: any) => d)])))]));
  const root = tx.mutations[0].term;
  assert(root.as_schema() === Mutations.OfUpsert.as_schema() && root.key.get() === "k");
  assert(root.then[0].next.as_schema() === Mutations.OfUpdate.as_schema());
  const other = store.Mutations.OfTransaction((t: any) => t.uid("t-2").mutations((a: any) => a.insert((e: any) => e.map((m: any) => m))));
  assert(other.mutations[0].term.as_schema() === Mutations.OfMap.as_schema());
  const kept = store.Mutations.OfTransaction((t: any) => t.uid("t-3").mutations((a: any) => a.insert((e: any) => e.update(root.then[0].next))));
  assert(kept.mutations[0].term === root.then[0].next); // a kind given an object links it
  raises(TypeError, () => store.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.uid])), // a member of no kind: which object?
    "link 'term' is a kind union: build through one of its kinds, map, reduce, partition, update, upsert");
  raises(TypeError, () => store.Mutations.OfTransaction((t: any) => t.mutations((a: any) => a.insert((e: any) => e.term((x: any) => x)))),
    "link 'term' is a kind union: build through one of its kinds");
  const seen: string[] = [];
  store.Mutations.OfTransaction((t: any) => t.mutations((a: any) => a.insert((e: any) => (seen.push(repr(e.upsert)), e.upsert((u: any) => u)))));
  assert(same(seen, ["<kind of link 'term'>"]));
}

## TXN-02 · Mappings for keyed adjacencies: each item an entry, its key the one property, its value a callback or given to the one member that takes it

In [ ]:
{
  assert(same([...Person.properties.keys()], ["name", "age"]) && Person.properties.get("age").schema === Int); // the schema builders'
  const Tagged = store.Schemas.OfObject((o: any) => o.properties({ tag: (t: any) => t.schema([Str]).description("a label") }));
  assert(Tagged.properties.get("tag").description === "a label");
  raises(TypeError, () => store.Schemas.OfAny((a: any) => a.schemas({ n: Int })),
    "'schemas' takes a mapping only where its relation has two links and one property");
  raises(TypeError, () => store.Schemas.OfObject((o: any) => o.properties({ x: "text" })), // its description and its naming pattern
    "more than one member of the entry takes 'text': give a callback");
  raises(TypeError, () => store.Schemas.OfObject((o: any) => o.properties({ x: 3n })), "no member of the entry takes 3: give a callback");
  const rootModule = store.module();
  store.Schemas.OfModule(rootModule, (m: any) => m.schemas({ Tagged })); // the link alone takes a schema
  assert(rootModule.schemas.get("Tagged") === Tagged);
  const book = store.singleton("crm.TheBook");
  store.crm.Book(book, (b: any) => b.contacts({ ada: (e: any) => e.name("Ada"), bob: (e: any) => e.name("Bob").age(36n) }));
  const [ada, bob] = [book.contacts({ key: "ada" }), book.contacts({ key: "bob" })]; // the proxies': each a contact, by its key
  assert(equal([...book.contacts].map((e: any) => [e.key.get(), e.contact.name.get()]), [["ada", "Ada"], ["bob", "Bob"]]) && bob.age.get() === 36n);
  store.crm.Person(ada, (c: any) => c.knows(new Map([[2020n, (e: any) => e.known(bob)]])));
  assert(equal([...ada.knows].map((e: any) => [e.since.get(), e.known]), [[2020n, bob]]));
  raises(TypeError, () => store.crm.Person(ada, (c: any) => c.knows(new Map([[2021n, bob]]))), // the link, and the linked person's own 'knows'
    "more than one member of the entry takes <crm.Person>: give a callback");
  raises(TypeError, () => store.crm.Book(book, (b: any) => b.contacts({ x: 1.5 })), "no member of the entry takes 1.5: give a callback");
  raises(TypeError, () => store.Mutations.OfTransaction((t: any) => t.mutations({ x: (e: any) => e })),
    "adjacency 'mutations' takes a mapping only where its relation has two links and one property");
  raises(TypeError, () => store.Schemas.OfModule(rootModule, (m: any) => m.schemas({ x: "x" })), "no member of the entry takes 'x': give a callback");
  raises(TypeError, () => store.Schemas.OfAny((a: any) => a.schemas("x")),
    "'schemas' takes an iterable of what it links, or of entries, a mapping, or a callback");
  // A link of a kind union, or of a schema it declares, takes an object of it; a callback chooses a kind.
  const note = store.crm.Note((n: any) => n.picks(new Map<bigint, unknown>([[1n, ada], [2n, (e: any) => e.note((m: any) => m.text("aside"))]]))
    .pins(new Map([[3n, book]])));
  assert(equal([...note.picks].map((e: any) => [e.rank.get(), e.pick.as_schema().name]), [[1n, "crm.Person"], [2n, "crm.Note"]]));
  assert(equal([...note.pins].map((e: any) => [e.at.get(), e.book]), [[3n, book]]));
  raises(TypeError, () => store.crm.Note((n: any) => n.odds(new Map([[1n, "zz"]]))), // an adjacency with no relation yet takes nothing
    "no member of the entry takes 'zz': give a callback");
  const outside = new B.OfObject.Builder().properties({ n: (t: any) => t.description("d") }).create(); // outside a store too
  assert(outside.properties.get("n").description === "d");
}

## TXN-03 · A transaction's expression: reference objects linked by relations, each term's pattern a schema object built through its link and held while its term is

In [ ]:
{
  const tx = store.Mutations.OfTransaction((t: any) => t.uid("t-9").mutations([(mu: any) => mu.upsert((u: any) => u.exists([
    (p: any) => p.symbols({ ada: Person }).description("whom")]))]));
  const pattern = tx.mutations[0].term.exists[0].pattern;
  assert(pattern instanceof S.OfPattern.Data && same([...pattern.symbols.keys()], ["ada"]) && pattern.description === "whom");
  assert(store.holds(pattern) && store.holds(pattern.symbols.get("ada"))); // weakly: it goes with its transaction (Decided, 94)
  raises(LookupError, () => Paths.of(pattern), "a pattern with no name of its own");
  const Edit = store.Schemas.OfPattern((p: any) => p.name("crm.Edit").symbols({ ada: Person }));
  store.Mutations.OfUpdate((u: any) => u.apply([(p: any) => p.pattern(Edit).description("edits")])); // linked, then updated
  assert(Edit.description === "edits");
  assert(store._meta_builder("crm.Person") === null && store._meta_builder("Schemas.Nope") === null);
  assert(repr(store._meta_builder("Schemas.OfPattern")) === "<builder of a pattern>");
  assert(store._meta_builder_class(Edit) === B.PatternBuilder);
}

## TXN-04 · A spec: written in the builders' notation over the frame's symbols, recorded as mutations, by symbol

In [ ]:
{
  const ref = (name: string): any => ({ ref: { symbol: name } });
  const pathOf = (target: unknown): any => new Map([["ref", new Map([["path", Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), Paths.of(target))]])]]);
  // (a path is a Map already: `map` keeps it)
  const link = (name: string, target: unknown): any => ({ as_link: { name, mutations: [{ set: target }] } });
  const prop = (name: string, value: unknown): any => ({ as_property: { name, mutations: [{ set: { value } }] } });
  const entries = (name: string, ...items: unknown[][]): any => ({ as_adjacency: { name, mutations: [{ insert: {
    entries: items.map((item) => ({ mutations: item })) } }] } });

  const Meet = store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person, bob: Person }).spec([
    (_: any) => _.bob.name("Bob").knows([_.ada]),
    (_: any) => _.ada.name("Ada").age(null).knows([(e: any) => e.known(_.bob).since(2020n)]),
  ]));
  assert(equal(Meet.spec, map([
    { bob: [prop("name", "Bob"), entries("knows", [link("known", ref("ada"))])] },
    { ada: [prop("name", "Ada"), { as_property: { name: "age", mutations: [{ clear: {} }] } },
      entries("knows", [link("known", ref("bob")), prop("since", 2020n)])] },
  ])));
  const book = store.singleton("crm.TheBook");
  const Owned = store.Schemas.OfPattern((p: any) => p.symbols({ bob: Person }).spec( // one callback; a path; a mapping
    (_: any) => _.bob.book({ bob: book }).knows((a: any) => a.insert((e: any) => e.known((n: any) => n.name("Dan")).since(1n)))));
  assert(equal(Owned.spec, map([{ bob: [
    entries("book", [link("book", { ref: pathOf(book).get("ref") }), prop("key", "bob")]),
    entries("knows", [link("known", { def: { schema: "crm.Person", mutations: [prop("name", "Dan")] } }), prop("since", 1n)]),
  ] }])));
  const Defines = store.Schemas.OfPattern((p: any) => p.symbols({ P: store.Schemas.OfObject }).spec([
    (_: any) => _.P.name("crm.Pet").ref(true).properties({ name: Str, legs: (t: any) => t.schema([Int]) }),
  ]));
  const propertySchema = (schema: unknown): any => ({ def: { schema: "Schemas.OfProperty", mutations: [
    entries("schema", [link("schema", { ref: pathOf(schema).get("ref") })])] } });
  assert(equal(Defines.spec, map([{ P: [prop("name", "crm.Pet"), prop("ref", true), entries(
    "properties", [link("property", propertySchema(Str)), prop("name", "name")],
    [link("property", propertySchema(Int)), prop("name", "legs")])] }])));
  const Terms = store.Schemas.OfPattern((p: any) => p.symbols({ t: store.Mutations.OfTransaction }).spec([
    (_: any) => _.t.uid("u").mutations([(e: any) => e.upsert((u: any) => u.key("k"))]), // a kind union: a kind defined
    (_: any) => _.t.mutations([(e: any) => e.term(_.t)]),
  ]));
  assert(equal(Terms.spec[0], map({ t: [prop("uid", "u"), entries("mutations", [link("term", { def: {
    schema: "Mutations.OfUpsert", mutations: [prop("key", "k")] } })])] })));
  assert(equal(Terms.spec[1], map({ t: [entries("mutations", [link("term", ref("t"))])] })));
  // Reading through the frame, and what a spec refuses.
  const seen: unknown[] = [];
  store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person }).spec([
    (_: any) => seen.push(repr(_), repr(_.ada), _.ada.name.get(), _.ada.as_schema() === Person,
      _.nope.as_schema(), _.ada.nope.as_schema(), _.ada.knows.as_schema().name,
      "_hidden" in _, "_hidden" in _.ada, "_hidden" in _.ada.name),
    (_: any) => _.ada.knows((a: any) => a.insert((e: any) => (seen.push(
      e.as_schema() === Knows.entry, "_hidden" in e, e.nope.as_schema()), e.known(_.ada)))),
    (_: any) => _.ada.knows([(e: any) => e.known((n: any) => { seen.push(repr(n)); })]),
  ]));
  assert(same(seen, ["<frame>", "<spec of ada>", null, true, null, null, "knows", false, false, false, true, false, null,
    "<spec of an object>"]));
  const Given = store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person, bob: Person }).spec([
    (_: any) => _.bob.knows(new Map([[2020n, "Dan"]])).book({ b: _.ada }), // a property of the linked object; an adjacency of it
    (_: any) => _.bob.knows([(e: any) => e.known((n: any) => n.name("Eve")).age(3n)]), // the object it defines, written on
  ]));
  assert(equal(Given.spec[0], map({ bob: [
    entries("knows", [link("known", { def: { schema: "crm.Person", mutations: [prop("name", "Dan")] } }), prop("since", 2020n)]),
    entries("book", [link("book", { def: { schema: "crm.Book", mutations: [entries("contacts", [link("contact", ref("ada"))])] } }),
      prop("key", "b")])] })));
  assert(equal(Given.spec[1], map({ bob: [entries("knows", [link("known", { def: { schema: "crm.Person", mutations: [
    prop("name", "Eve"), prop("age", 3n)] } })])] })));
  const note = store.crm.Note((n: any) => n);
  for (const [spec, error, says] of [
    [(_: any) => _.x.name("x"), TypeError, "symbol 'x' has no schema: a spec writes what its object's schema declares"],
    [(_: any) => _.ada.knows(3n), TypeError, "adjacency 'knows' takes an iterable of entries or of what it links, a mapping"],
    [(_: any) => _.ada.book({ a: 1.5 }), TypeError, "no member of the entry takes 1.5: give a callback"],
    [(_: any) => _.ada.knows([(e: any) => e.known(_.ada).name("Z")]), ValueError,
      "link 'known' links an object a spec does not define: write its members in its own spec"],
    [(_: any) => _.ada.knows([(e: any) => e.known(note)]), ValueError,
      "a counted 'crm.Note' has no path: a spec refers to it through a symbol"],
    [(_: any) => _.ada.meets([(e: any) => e]), TypeError,
      "adjacency 'meets': entries of crm.Meets have more than two links; give each entry's links"],
    [(_: any) => _.ada.trio([(e: any) => e]), TypeError, "adjacency 'trio': entries of its relation have more than two links"],
    [(_: any) => _.ada.knows(new Map([[2020n, _.ada]])), TypeError, "more than one member of the entry takes <spec of ada>: give a callback"],
    [(_: any) => _.ada.meets(new Map([[1n, _.ada]])), TypeError,
      "adjacency 'meets' takes a mapping only where its relation has two links and one property"],
    [(_: any) => _.ada.knows([(e: any) => e.known((n: any) => n.knows([(f: any) => f.known(n)]))]), ValueError,
      "an object a spec defines is linked where it is defined"],
    [(_: any) => _.t.mutations([(e: any) => e.term((x: any) => x)]), TypeError,
      "link 'term' is a kind union: define through one of its kinds, map, reduce, partition, update, upsert"],
    [(_: any) => _.t.mutations([(e: any) => e.uid("u")]), TypeError,
      "link 'term' is a kind union: define through one of its kinds, map, reduce, partition, update, upsert"],
  ] as [(_: any) => unknown, Function, string][]) {
    raises(error, () => store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person, x: (t: any) => t,
      t: store.Mutations.OfTransaction }).spec([spec])), says);
  }
  const entry: any[] = [];
  store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person }).spec([(_: any) => _.ada.knows([(e: any) => (entry.push(e), e.known(_.ada))])]));
  raises(AttributeError, () => { entry[0].since = 1n; }); // a member is written by calling it
  raises(TypeError, () => new B.PatternBuilder().spec([(_: any) => _]), "a spec is written in a store, whose schemas its symbols name");
  raises(ReadOnlyError, () => new B.PatternBuilder(Meet, false, store).spec([(_: any) => _]));
}